In [1]:
import pandas as pd
import polars as pl
import pandera.polars as pa
import time
from memory_profiler import memory_usage

pd.set_option('display.max_columns', None)

# 1. CONTRATO DE DADOS (PANDERA)
class TaxiSchema(pa.DataFrameModel):
    """
    Contrato focado nas colunas de agregação e em variáveis financeiras críticas.
    """
    # Regras de Negócio
    passenger_count: float = pa.Field(ge=0.0, nullable=True) # Usamos float pois CSVs com nulos forçam esse tipo inicialmente
    PULocationID: int = pa.Field(nullable=False)

    # Validação Financeira
    fare_amount: float = pa.Field(nullable=False)
    total_amount: float = pa.Field(nullable=True)
    payment_type: float = pa.Field(isin=[1, 2, 3, 4, 5, 6], nullable=True) # Domínio de tipos de pagamento (1 a 6)

    class Config:
        strict = False # Permite que as outras 15 colunas do arquivo passem sem checagem
        coerce = True  # Força a conversão de tipos quando seguro (ex: int para float)

# 2. FUNÇÕES DE PROCESSAMENTO
def process_pandas(path: str):
    df = pd.read_csv(path)
    grouped_df = (
        df.query("passenger_count > 1")
        .groupby(by="PULocationID")
        .agg({"fare_amount": "mean"})
        .sort_values(by="fare_amount", ascending=False)
        .reset_index()
    )
    return grouped_df

def process_polars(path: str):
    # 1. Inicia o plano de execução (Lazy)
    df = pl.scan_csv(path)

    # 2. Injeta o contrato de dados no plano
    df_validado = TaxiSchema.validate(df)

    # 3. Aplica as transformações e engatilha o processamento (collect)
    grouped_df = (
        df_validado.filter(pl.col("passenger_count") > 1)
        .group_by("PULocationID")
        .agg(pl.col("fare_amount").mean())
        .sort(by="fare_amount", descending=True)
        .collect()
    )
    return grouped_df

# 3. EXECUÇÃO DO BENCHMARK VALIDADO
path = 'data/test_data.csv'

print("Iniciando Benchmark com Validação de Contrato...")

print("\nRodando Polars (Com Pandera)...")
start_time = time.time()
try:
    df_pl = process_polars(path)
    polars_time = time.time() - start_time
    polars_memory = memory_usage((process_polars, (path,)), max_usage=True)

    print("✅ Contrato validado com sucesso!")
    print(f"Tempo: {polars_time:.2f}s | RAM Máxima: {polars_memory:.0f}MB")
    display(df_pl.head(5))

except pa.errors.SchemaErrors as e:
    print("❌ Falha no Contrato de Dados! Processamento abortado.")
    print(e.failure_cases)

print("\nRodando Pandas (Baseline sem validação)...")
start_time = time.time()
df_pd = process_pandas(path)
pandas_time = time.time() - start_time
pandas_memory = memory_usage((process_pandas, (path,)), max_usage=True)
print(f"Tempo: {pandas_time:.2f}s | RAM Máxima: {pandas_memory:.0f}MB")

Iniciando Benchmark com Validação de Contrato...

Rodando Polars (Com Pandera)...
✅ Contrato validado com sucesso!
Tempo: 2.96s | RAM Máxima: 1293MB


PULocationID,fare_amount
i64,f64
176,206.7
44,149.0
253,128.620952
23,113.46
221,100.0



Rodando Pandas (Baseline sem validação)...


/var/folders/rm/s36g6dxn0rvfklb433ww803c0000gn/T/ipykernel_4758/3734982429.py:29: DtypeWarning: Columns (0: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
/var/folders/rm/s36g6dxn0rvfklb433ww803c0000gn/T/ipykernel_4758/3734982429.py:29: DtypeWarning: Columns (0: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


Tempo: 15.48s | RAM Máxima: 2017MB
